Bing Chen 

ST 554 Project Part 2 

3/19/2026

# Introduction 

In this project, we developed a PySpark-based data validation class named `SparkDataCheck`. Our class is designed to facilitate data validation and exploration. After defining the class and its associated methods, we will apply it to some real-world dataset here to perform various data quality checks, such as missing value detection, numerical range validation, and etc. 


Now, let's import the class and start a spark session. 

In [14]:
from SparkDataCheck import SparkDataCheck
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder \
    .appName("SparkDataCheck") \
    .getOrCreate()



Next, let's create an instance of the class from a `.csv` file. 

In [2]:
air = SparkDataCheck.from_csv(
    spark,
    "air.csv"
)

air.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-20 21:00:00|   2.2|       137

26/03/20 14:25:02 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Using this air quality data, let's see some examples of how to use methods in this class. Recall that in this dataset, missing values are tagged with -200 value, so let's replace them with `None` first. 

In [4]:
air.df = air.df.na.replace(-200, None)

## Check if a numeric variable is within a specified range. 

Next, we might want to check if `PT08.S1(CO)` values specifically are all above 0. 


In [5]:
air.within_range("PT08.S1(CO)", lower=0, upper = None).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|                true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 14:25:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Let's see if the values are all below 2000. 

In [9]:
air.within_range("PT08.S1(CO)", lower=None, upper = 1300).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 14:27:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


How about between 0 to 1500?

In [7]:
air.within_range("PT08.S1(CO)", lower=0, upper = 10).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 14:26:31 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


What if we accidentally passed through a string variable? 

In [10]:
air.within_range("Date", lower=0, upper = None).df.show()

Column 'Date' is not numeric. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|  

26/03/20 14:28:03 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Nice! It prints out message noting that `Date` is not a numeric column and returns the original data frame. 

What if we forgot to pass on any bounds?

In [11]:
air.within_range("PT08.S1(CO)").df.show()

At least one of 'lower' or 'upper' must be provided. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       14

26/03/20 14:29:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


So, it prints out that at least one bound need to be supplied. Great! 

## Check if a categorical variable is within specified levels. 

Say we want to classify `PT08.S1(CO)` into 3 levels to allow users to quickly understand if air quality is "Safe" or "Hazardous" without knowing technical sensor specs. Consider the 3 ranks below: 

 - Low: Readings below 970
 - Mid: Readings between 970 and 1,180
 - High: Readings above 1,180
 
 So, let's create a new variable for the transformed `PT08.S1(CO)`. 


In [15]:
air.df = air.df.withColumn(
    "PT08_S1(CO)_cat",
    F.when(F.col("`PT08.S1(CO)`") < 970, "Low")
     .when((F.col("`PT08.S1(CO)`") >= 970) & (F.col("`PT08.S1(CO)`") <= 1180), "Mid")
     .when(F.col("`PT08.S1(CO)`") > 1180, "High")
     .otherwise(None)
)

Let's quickly check the results. 

In [21]:
air.df.select("`PT08.S1(CO)`","PT08_S1(CO)_cat").show(10)

+-----------+---------------+
|PT08.S1(CO)|PT08_S1(CO)_cat|
+-----------+---------------+
|       1360|           High|
|       1292|           High|
|       1402|           High|
|       1376|           High|
|       1272|           High|
|       1197|           High|
|       1185|           High|
|       1136|            Mid|
|       1094|            Mid|
|       1010|            Mid|
+-----------+---------------+
only showing top 10 rows


Looks like it's working correctly. Let's try out our method to see if all values are within the levels. 

In [23]:
air.within_levels("PT08.S1(CO)", levels = ["Low", "Mid", "High"]).df.show()

Column 'PT08.S1(CO)' is not a string column.
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|                     true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4

26/03/20 14:42:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Oops! Accidentally passed through the numeric column of `PT08.S1(CO)` instead of the categorical column we just created! So, let's do that again. 

In [22]:
air.within_levels("PT08_S1(CO)_cat", levels = ["Low", "Mid", "High"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|                     true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92| 

26/03/20 14:42:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Looks good! Let's try another variable.  We can `Absolute Humidity` into 

- Dry: $ < 7 g/m^3$
- Optimal:  $ 7 ~ 11 g/m^3$
- Humid:  $ > 11 g/m^3 $

In [25]:
air.df = air.df.withColumn(
    "abs_humidity_cat",
    F.when(F.col("AH") < 7, "Dry")
     .when((F.col("AH") >= 7) & (F.col("AH") <= 11), "Optimal")
     .when(F.col("AH") > 11, "Humid")
     .otherwise(None)
)

In [30]:
air.within_levels("abs_humidity_cat", levels = ["Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 14:53:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Why is there so many `false`? Oops! Accidentally missed out a level! 

In [26]:
air.within_levels("abs_humidity_cat", levels = ["Dry", "Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 14:51:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Now it's looking good. 

## Check Missingness 

When we first look at a dataset, we always want to know how complete is the data, are there any missing values, how are they coded? From the data description, we know that missing values are coded -200 and we already replaced that with `None`. So now, let's check where a value is missing for some variables. 

In [31]:
air.is_missing("`PT08.S1(CO)`").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|  

26/03/20 14:57:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [32]:
air.is_missing("AH").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|  

26/03/20 14:59:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [33]:
air.is_missing("NMHC(GT)").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|  0|3/10/2004|2026-03-20 18

26/03/20 15:00:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [34]:
air.is_missing("Date").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|Date_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-----------------

26/03/20 15:00:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


SyntaxError: invalid syntax (4238376967.py, line 1)